In [ ]:
import pandas as pd
import os
from pathlib import Path
from collections import deque

# Set up project paths
project_root = Path('/workspaces/CellTreeBench')
data_dir = project_root / 'data' / 'celegans_small' / 'raw'
out_dir = project_root / 'data' / 'celegans_small' / 'P0' / 'tree_building' / 'tree_df'

# Ensure output directory exists
out_dir.mkdir(parents=True, exist_ok=True)

print(f"Input directory: {data_dir}")
print(f"Output directory: {out_dir}")

# Load Cell Metadata

In [ ]:
file_name = data_dir / "metadata.csv"
metadata_df = pd.read_csv(file_name)
print(f"Loaded metadata from: {file_name}")
metadata_df

/tmp/ipykernel_2769732/1717842517.py:2: DtypeWarning: Columns (2,5,6,7,8,15) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata_df = pd.read_csv(file_name)


,cell,n.umi,time.point,batch,Size_Factor,cell.type,cell.subtype,plot.cell.type,lineage_packer,embryo.time,smoothed.embryo.time,embryo.time.bin,genotype,dataset,jt.cell.subtype2,joint_lineage
0,AAACCCAAGCAGCCCT-Ce_ceh9_1,1468,300_minutes,Ce_ceh9_1_batch,0.859795,NaN,NaN,NaN,NaN,200.0,210.0,210_270,ceh-9,Ce_ceh9_300_minutes,early_pharynx,NaN
1,AAACCCAAGCTAGATA-Ce_ceh9_1,1823,300_minutes,Ce_ceh9_1_batch,1.072126,NaN,NaN,NaN,NaN,160.0,170.0,170_210,ceh-9,Ce_ceh9_300_minutes,early_embryo,NaN
2,AAACCCAAGGCGCTTC-Ce_ceh9_1,18285,300_minutes,Ce_ceh9_1_batch,10.637488,NaN,NaN,NaN,NaN,100.0,60.0,lt_100,ceh-9,Ce_ceh9_300_minutes,germline,NaN
3,AAACCCAAGTGATCGG-Ce_ceh9_1,2289,300_minutes,Ce_ceh9_1_batch,1.337976,NaN,NaN,NaN,NaN,350.0,370.0,330_390,ceh-9,Ce_ceh9_300_minutes,unassigned,NaN
4,AAACCCAGTCGTATGT-Ce_ceh9_1,2932,300_minutes,Ce_ceh9_1_batch,1.716681,NaN,NaN,NaN,NaN,370.0,390.0,390_450,ceh-9,Ce_ceh9_300_minutes,hyp4_hyp5_hyp6,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255022,TCTGAGACATGTCGAT-b02,585,mixed,Murray_b02,0.337984,Rectal_gland,Rectal_gland,Rectal_gland,NaN,700.0,360.0,330_390,wt,Murray_b02,B_F_K_Kp_U_Y,NaN
255023,TCTGAGACATGTCTCC-b02,510,mixed,Murray_b02,0.300172,NaN,NaN,NaN,NaN,470.0,460.0,450_510,wt,Murray_b02,g2,NaN
255024,TGGCCAGCACGAAGCA-b02,843,mixed,Murray_b02,0.490397,NaN,NaN,NaN,NaN,470.0,450.0,450_510,wt,Murray_b02,CEP,NaN
255025,TGGCGCACAGGCAGTA-b02,636,mixed,Murray_b02,0.368816,NaN,NaN,NaN,NaN,350.0,350.0,330_390,wt,Murray_b02,PVP,NaN


# Load the Mapping of Concept Lineage to Molecular Lineage

In [ ]:
file_name = data_dir / "CellTable.csv"
cell_table = pd.read_csv(file_name)
print(f"Loaded CellTable from: {file_name}")
cell_table

,Lineage,Cell,SimpleName,Description,Time,TerminalDatasetName,ProgenitorDatasetName,Missing,TerminalPackerName,Parent,ParentDatasetName,CellClass,br_time,d_time,level,MergedDatasetName,height
0,AB,progenitor,progenitor,NaN,3.0,NaN,NaN,Yes,NaN,P0,NaN,progenitor,1,19.0,1,NaN,18
1,ABa,progenitor,progenitor,NaN,19.0,NaN,NaN,Yes,NaN,AB,NaN,progenitor,19,43.0,2,NaN,24
2,ABal,progenitor,progenitor,NaN,43.0,NaN,NaN,Yes,NaN,ABa,NaN,progenitor,43,56.0,3,NaN,13
3,ABala,progenitor,progenitor,NaN,56.0,NaN,ABaxx,No,NaN,ABal,NaN,progenitor,56,75.0,4,ABaxx,19
4,ABalaa,progenitor,progenitor,NaN,75.0,NaN,NaN,Yes,NaN,ABala,ABaxx,progenitor,75,103.0,5,NaN,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1331,P4,progenitor,progenitor,NaN,75.0,NaN,NaN,Yes,NaN,P3,NaN,progenitor,75,142.0,4,NaN,67
1332,Z3,Z3,Z3,Germ line precursor cell,142.0,germline,NaN,No,Germline,P4,NaN,Germline,142,NaN,5,germline,578
1333,MSappaap,Z4,Z4,Somatic gonad precursor cell,263.0,Z1_Z4,MSxppaap,No,Z1_Z4,MSappaa,MSxppaa,Mesoderm,263,NaN,9,Z1_Z4,457
1334,ABprapappppa,progenitor,progenitor,NaN,NaN,NaN,NaN,Yes,NaN,ABprapapppp,NaN,NaN,395,480.0,11,NaN,325


# Identify the roots of subtrees of the Molecular Lineage

In [15]:
# Check all the root liineages
# ProgenitorDatasetName should be not Nan
# ParentDatasetName should be Nan
root_lineages = cell_table[cell_table["ProgenitorDatasetName"].notna() & cell_table["ParentDatasetName"].isna()][["Lineage", "ProgenitorDatasetName", "Parent"]]
# Drop rows if there is "/" in "ProgenitorDatasetName"
root_lineages = root_lineages[~root_lineages["ProgenitorDatasetName"].str.contains("/")]
root_lineages = root_lineages.reset_index(drop=True)
print(root_lineages['ProgenitorDatasetName'].unique())
root_lineages

['ABaxx' 'ABalapx' 'ABalpax' 'ABalppa' 'ABaraax' 'ABarapa' 'ABpxax'
 'ABpxp' 'Cx' 'Dx' 'Exx' 'MSx']


,Lineage,ProgenitorDatasetName,Parent
0,ABala,ABaxx,ABal
1,ABalapa,ABalapx,ABalap
2,ABalapp,ABalapx,ABalap
3,ABalp,ABaxx,ABal
4,ABalpaa,ABalpax,ABalpa
5,ABalpap,ABalpax,ABalpa
6,ABalppa,ABalppa,ABalpp
7,ABara,ABaxx,ABar
8,ABaraaa,ABaraax,ABaraa
9,ABaraap,ABaraax,ABaraa


In [16]:
root_lineages[['ProgenitorDatasetName', 'Parent']].drop_duplicates()

,ProgenitorDatasetName,Parent
0,ABaxx,ABal
1,ABalapx,ABalap
4,ABalpax,ABalpa
6,ABalppa,ABalpp
7,ABaxx,ABar
8,ABaraax,ABaraa
10,ABarapa,ABarap
12,ABpxax,ABpla
14,ABpxp,ABpl
15,ABpxax,ABpra


# Export the subtree of the molecular lineage

In [37]:
subtree_node_list = root_lineages[['ProgenitorDatasetName']].drop_duplicates()['ProgenitorDatasetName'].tolist()
subtree_node_list

['ABaxx',
 'ABalapx',
 'ABalpax',
 'ABalppa',
 'ABaraax',
 'ABarapa',
 'ABpxax',
 'ABpxp',
 'Cx',
 'Dx',
 'Exx',
 'MSx']

In [84]:
root_molecular_tree = subtree_node_list[7]
print(root_molecular_tree)
# Based on the Molecualr Tree, we can find the root of the lineage tree
def get_lineage_node(cell_table, root_molecular):
    cur_lineage = "ProgenitorDatasetName"
    par_lineage = "ParentDatasetName"
    res = []
    for index, row in cell_table.iterrows():
        if row[cur_lineage] == root_molecular:
            res.append(row)
    return pd.DataFrame(res)

df = get_lineage_node(cell_table, root_molecular_tree)
df

ABpxp


,Lineage,Cell,SimpleName,Description,Time,TerminalDatasetName,ProgenitorDatasetName,Missing,TerminalPackerName,Parent,ParentDatasetName,CellClass,br_time,d_time,level,MergedDatasetName,height
588,ABplp,progenitor,progenitor,NaN,56.0,NaN,ABpxp,No,NaN,ABpl,NaN,progenitor,56,75.0,4,ABpxp,19
838,ABprp,progenitor,progenitor,NaN,56.0,NaN,ABpxp,No,NaN,ABpr,NaN,progenitor,56,75.0,4,ABpxp,19


In [85]:
# Based on the Molecualr Tree, we can find the root of the lineage tree
def get_lineage_descent(cell_table, root_molecular):
    cur_lineage = "ProgenitorDatasetName"
    par_lineage = "ParentDatasetName"
    res = []
    for index, row in cell_table.iterrows():
        if row[cur_lineage] == root_molecular:
            print(root_molecular)
            res.append(row)

    par_list = [root_molecular]
    while len(par_list) > 0:
        the_par = par_list.pop(0)
        for index, row in cell_table.iterrows():
            if row[par_lineage] == the_par:
                # print(row[cur_lineage])
                # Filter 2: If the node cannot mpa to the molecular tree, we will ignore it
                if pd.isna(row[cur_lineage]):
                    continue
                # filter 1: if row[cur_lineage] has "/", ignore it
                if "/" in row[cur_lineage]:
                    continue
                
                res.append(row)
                
                # If the node is not in the par_list, we will add it to the par_list to further explore
                if row[cur_lineage] not in par_list:
                    par_list.append(row[cur_lineage])
        # print(len(par_list))
    return pd.DataFrame(res)

df = get_lineage_descent(cell_table, root_molecular_tree)
df

ABpxp
ABpxp


,Lineage,Cell,SimpleName,Description,Time,TerminalDatasetName,ProgenitorDatasetName,Missing,TerminalPackerName,Parent,ParentDatasetName,CellClass,br_time,d_time,level,MergedDatasetName,height
588,ABplp,progenitor,progenitor,NaN,56.0,NaN,ABpxp,No,NaN,ABpl,NaN,progenitor,56,75.0,4,ABpxp,19
838,ABprp,progenitor,progenitor,NaN,56.0,NaN,ABpxp,No,NaN,ABpr,NaN,progenitor,56,75.0,4,ABpxp,19
589,ABplpa,progenitor,progenitor,NaN,75.0,NaN,ABpxpa,No,NaN,ABplp,ABpxp,progenitor,75,103.0,5,ABpxpa,28
654,ABplpp,progenitor,progenitor,NaN,75.0,NaN,ABpxpp,No,NaN,ABplp,ABpxp,progenitor,75,104.0,5,ABpxpp,29
839,ABprpa,progenitor,progenitor,NaN,75.0,NaN,ABpxpa,No,NaN,ABprp,ABpxp,progenitor,75,103.0,5,ABpxpa,28
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
350,ABprppppapp,DVA,DVA,"Ring interneuron, cell body in dorsorectal gan...",291.0,DVA,ABprppppapp,No,DVA,ABprppppap,ABpxppppap,Non-ciliated neurons,291,NaN,10,DVA,429
393,ABplppppapp,F,F,"Rectal cell, blast cell in male",292.0,B_F_K_Kp_U_Y,ABplppppapx,No,B,ABplppppap,ABpxppppap,Rectal cells,292,NaN,10,B_F_K_Kp_U_Y,428
1245,ABplppppapa,U,U,"Rectal cell, postembryonic blast cell in male",292.0,B_F_K_Kp_U_Y,ABplppppapx,No,F_U,ABplppppap,ABpxppppap,Rectal cells,292,NaN,10,B_F_K_Kp_U_Y,428
1119,ABplppaaaapp,RICL,RIC,Ring interneuron,400.0,RIC,ABpxppaaaapp,No,RIC,ABplppaaaap,ABpxppaaaap,Non-ciliated neurons,405,NaN,11,RIC,315


In [ ]:
from collections import deque
tree_df = df[["ProgenitorDatasetName", "ParentDatasetName"]].drop_duplicates()
# Rename the columns as Linage and Parent
tree_df.columns = ["Lineage", "Parent"]
# Add a new column to store the level of the lineage
tree_df["Level"] = 0
# Create a dictionary for parent lookup from the DataFrame
parent_dict = tree_df.set_index('Lineage')['Parent'].to_dict()

# Initialize the level dictionary and queue for BFS
level_dict = {}
queue = deque()

for lineage, parent in parent_dict.items():
    if pd.isna(parent):
        queue.append((lineage, 0)) 
        
# BFS to calculate levels
while queue:
    current_node, current_level = queue.popleft()
    level_dict[current_node] = current_level

    # Find all children of the current node
    children = [lineage for lineage, parent in parent_dict.items() if parent == current_node]
    for child in children:
        if child not in level_dict:  # To avoid processing the same node twice
            queue.append((child, current_level + 1))  
# Assign levels to the DataFrame
tree_df['Level'] = tree_df['Lineage'].map(level_dict)

# For each row in in tree_df Lineage column, 
# check how many rows in metadata_df have elements in metadata_df$Lineage has the same value as in tree_df$Lineage
# and save in tree_df$n_cells
n_cells = []
for lineage in tree_df.Lineage:
    n_cells.append(metadata_df[metadata_df.lineage_packer == lineage].shape[0])
tree_df['n_cells'] = n_cells

file_name = out_dir / f"tree_df-{root_molecular_tree}.csv"
print(f"Save the tree_df to {file_name}")
tree_df.to_csv(file_name, index=False)
tree_df                  

Save the tree_df to /workspaces/1-phydist/main/data/celegans/raw/tree_df/tree_df-ABpxp.csv


,Lineage,Parent,Level,n_cells
588,ABpxp,NaN,0,12
589,ABpxpa,ABpxp,1,13
654,ABpxpp,ABpxp,1,9
590,ABpxpaa,ABpxpa,2,10
625,ABpxpap,ABpxpa,2,9
...,...,...,...,...
1093,ABpxppppaaa,ABpxppppaa,6,55
236,ABprppppapa,ABpxppppap,6,35
350,ABprppppapp,ABpxppppap,6,48
393,ABplppppapx,ABpxppppap,6,46


In [89]:
from ete3 import Tree

def create_tree(tree_df):
    # Create the root of the tree
    node_dict = {}
    root_name = tree_df["Lineage"].iloc[0]
    tree = Tree(name=root_name)
    tree.add_feature("n_cells", tree_df["n_cells"].iloc[0])

    node_dict[root_name] = tree

    # Create nodes and arrange by parent
    for index, row in tree_df.iterrows():
        lineage_name = row["Lineage"]
        parent_name = row["Parent"]
        # Skip nodes with n_cells = 0
        # if row["n_cells"] == 0 or lineage_name == root_name:
        if lineage_name == root_name:
            continue

        # Add current node if it doesn't exist
        if lineage_name not in node_dict:
            node = Tree(name=lineage_name)
            node.add_feature("n_cells", row["n_cells"])
            node_dict[lineage_name] = node

        # Ensure parent node exists
        if parent_name not in node_dict:
            # node_dict[parent_name] = Tree(name=parent_name)
            print(f"ERR: Parent node {parent_name} not found for {lineage_name}.")

        # Attach the current node to its parent
        node_dict[parent_name].add_child(node_dict[lineage_name])
    return tree

In [ ]:
molecular_tree = create_tree(tree_df)
print(len(molecular_tree.get_leaf_names()))
print(molecular_tree.get_ascii(attributes=["name", "n_cells"]))
# Save the get_ascii to a file
file_name = out_dir / f"{root_molecular_tree}.txt"
with open(file_name, "w") as f:
    f.write(molecular_tree.get_ascii(attributes=["name"]))
print(f"Saved tree ASCII to: {file_name}")

file_name = out_dir / f"{root_molecular_tree}-ncells.txt"
with open(file_name, "w") as f:
    f.write(molecular_tree.get_ascii(attributes=["name", "n_cells"]))
print(f"Saved tree ASCII with cell counts to: {file_name}")

39

                                                       /ABpxpaaaaa, 195-ABpxpaaaaap, 180
                                          /ABpxpaaaa, 62
                                         |             \ABpxpaaaap, 176-ABpxpaaaapa, 65
                              /ABpxpaaa, 42
                             |           |             /-ABpxpaaapa, 115
                             |            \ABpxpaaap, 50
                             |                         \-ABpxpaaapp, 207
                             |
                   /ABpxpaa, 10                                       /-ABpxpaapaap, 114
                  |          |                         /ABpxpaapaa, 212
                  |          |            /ABpxpaapa, 72              \-ABpxpaapaaa, 124
                  |          |           |            |
                  |          |           |             \ABpxpaapap, 126-ABpxpaapapa, 111
                  |           \ABpxpaap, 37
                  |                      |   

In [ ]:
# Process all remaining subtrees
processed_trees = {}

for root_molecular_tree in subtree_node_list:
    print(f"\n{'='*50}")
    print(f"Processing: {root_molecular_tree}")
    print(f"{'='*50}")
    
    # Get lineage descent for this root
    df = get_lineage_descent(cell_table, root_molecular_tree)
    
    if df.empty:
        print(f"No data found for {root_molecular_tree}, skipping...")
        continue
    
    # Create tree dataframe with levels and cell counts
    tree_df = df[["ProgenitorDatasetName", "ParentDatasetName"]].drop_duplicates()
    tree_df.columns = ["Lineage", "Parent"]
    tree_df["Level"] = 0
    
    # Create parent dictionary and calculate levels using BFS
    parent_dict = tree_df.set_index('Lineage')['Parent'].to_dict()
    level_dict = {}
    queue = deque()
    
    for lineage, parent in parent_dict.items():
        if pd.isna(parent):
            queue.append((lineage, 0))
    
    while queue:
        current_node, current_level = queue.popleft()
        level_dict[current_node] = current_level
        
        children = [lineage for lineage, parent in parent_dict.items() if parent == current_node]
        for child in children:
            if child not in level_dict:
                queue.append((child, current_level + 1))
    
    tree_df['Level'] = tree_df['Lineage'].map(level_dict)
    
    # Add cell counts
    n_cells = []
    for lineage in tree_df.Lineage:
        n_cells.append(metadata_df[metadata_df.lineage_packer == lineage].shape[0])
    tree_df['n_cells'] = n_cells
    
    # Save tree dataframe
    file_name = out_dir / f"tree_df-{root_molecular_tree}.csv"
    tree_df.to_csv(file_name, index=False)
    print(f"Saved tree dataframe to: {file_name}")
    
    # Create molecular tree
    molecular_tree = create_tree(tree_df)
    n_leaves = len(molecular_tree.get_leaf_names())
    processed_trees[root_molecular_tree] = n_leaves
    
    print(f"Number of leaves: {n_leaves}")
    
    # Save tree ASCII representations
    file_name = out_dir / f"{root_molecular_tree}.txt"
    with open(file_name, "w") as f:
        f.write(molecular_tree.get_ascii(attributes=["name"]))
    
    file_name = out_dir / f"{root_molecular_tree}-ncells.txt"
    with open(file_name, "w") as f:
        f.write(molecular_tree.get_ascii(attributes=["name", "n_cells"]))
    
    print(f"Saved ASCII representations for {root_molecular_tree}")

print(f"\n{'='*60}")
print("PROCESSING COMPLETE!")
print(f"{'='*60}")
print("Summary of processed subtrees:")
for tree_name, leaf_count in processed_trees.items():
    print(f"  {tree_name}: {leaf_count} leaves")
print(f"\nTotal subtrees processed: {len(processed_trees)}")
print(f"Files saved to: {out_dir}")
